In [ ]:

# ================================
# Import Libraries
# ================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics


# ================================
# Load Dataset
# ================================

bank_df = pd.read_csv('bank.csv')
print(bank_df.head(5))


# ================================
# Separate Classes
# ================================

bank_subscribed_no = bank_df[bank_df.subscribed == 'no']
bank_subscribed_yes = bank_df[bank_df.subscribed == 'yes']


# ================================
# Upsampling Minority Class
# ================================

df_minority_upsampled = resample(
    bank_subscribed_yes,
    replace=True,
    n_samples=2000,
    random_state=42
)


# ================================
# Combine Data
# ================================

new_bank_df = pd.concat([bank_subscribed_no, df_minority_upsampled])


# ================================
# Feature Selection
# ================================

X_features = list(new_bank_df.columns)

# Remove target column
X_features.remove('subscribed')


# ================================
# Encoding
# ================================

encoded_bank_df = pd.get_dummies(
    new_bank_df[X_features],
    drop_first=True
)

X = encoded_bank_df

# ✅ FIX: Handle missing values
X = X.fillna(0)


# ================================
# Target Variable
# ================================

Y = new_bank_df.subscribed.map(lambda x: int(x == 'yes'))


# ================================
# Train Test Split
# ================================

train_X, test_X, train_y, test_y = train_test_split(
    X, Y,
    test_size=0.3,
    random_state=42
)


# ================================
# Logistic Regression Model
# ================================

logit = LogisticRegression(max_iter=10000)
logit.fit(train_X, train_y)

pred_y = logit.predict(test_X)


# ================================
# Confusion Matrix
# ================================

def draw_cm(actual, predicted):

    cm = metrics.confusion_matrix(actual, predicted, labels=[1, 0])

    sn.heatmap(
        cm,
        annot=True,
        fmt='.2f',
        xticklabels=["Subscribed", "Not Subscribed"],
        yticklabels=["Subscribed", "Not Subscribed"]
    )

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


draw_cm(test_y, pred_y)

print("\nClassification Report:\n")
print(metrics.classification_report(test_y, pred_y))


# ================================
# Prediction Probabilities
# ================================

predict_proba_df = pd.DataFrame(logit.predict_proba(test_X))
print(predict_proba_df.head())


# ================================
# Test Results DataFrame
# ================================

test_results_df = pd.DataFrame({'actual': test_y})
test_results_df = test_results_df.reset_index(drop=True)

test_results_df['chd_1'] = predict_proba_df.iloc[:, 1]

print(test_results_df.head(5))


# ================================
# ROC AUC Score
# ================================

auc_score = metrics.roc_auc_score(
    test_results_df.actual,
    test_results_df.chd_1
)

print("\nAUC Score:", round(float(auc_score), 2))


# ================================
# ROC Curve Function
# ================================

def draw_roc_curve(model, test_X, test_y):

    test_results_df = pd.DataFrame({'actual': test_y})
    test_results_df = test_results_df.reset_index(drop=True)

    predict_proba_df = pd.DataFrame(model.predict_proba(test_X))

    test_results_df['chd_1'] = predict_proba_df.iloc[:, 1]

    fpr, tpr, thresholds = metrics.roc_curve(
        test_results_df.actual,
        test_results_df.chd_1
    )

    auc_score = metrics.roc_auc_score(
        test_results_df.actual,
        test_results_df.chd_1
    )

    plt.figure(figsize=(8, 6))

    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % auc_score)
    plt.plot([0, 1], [0, 1], 'k--')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    plt.xlabel('False Positive Rate or [1 - True Negative Rate]')
    plt.ylabel('True Positive Rate')

    plt.title('Receiver Operating Characteristic')

    plt.legend(loc="lower right")
    plt.show()

    return auc_score, fpr, tpr, thresholds


# ================================
# Call ROC Function
# ================================

_, _, _, _ = draw_roc_curve(logit, test_X, test_y)
